In [1]:
import torch 
from torchvision import datasets, transforms
import numpy as np

In [2]:
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
])
mnist_data = datasets.MNIST(root='./data',train=True,download=True,transform=transform)
img_tensor, label = mnist_data[0]
img_np = img_tensor.numpy()

## discriminator building

In [3]:
# manual convolution 2d implementation
def discriminator_conv2d(image,kernels,stride=2,padding=1):
        out_c,in_c,kh,kw = kernels.shape
        img_c,img_h,img_w = image.shape
        image_padded = np.pad(image,((0,0),(padding,padding),(padding,padding)),mode = 'constant')
        p_h,p_w = image_padded.shape[1],image_padded.shape[2]
        out_h = (p_h-kh)//stride+1
        out_w = (p_w-kw)//stride+1
        output = np.zeros((out_c,out_h,out_w))
        for f in range(out_c):
            for y in range(out_h):
                  for x in range(out_w):
                        y_start = y*stride
                        x_start = x*stride
                        patch = image_padded[:,y_start:y_start+kh,x_start:x_start+kw]
                        output[f,y,x] = np.sum(patch*kernels[f])
        return output


In [4]:
# activation functions
def leaky_relu(x,alpha = 0.2):
    return np.where(x>0,x,x*alpha)
def relu(x):
    return np.maximum(0,x)
def sigmoid(x):
    return 1/(1+np.exp(-x))


##### dryrun of discriminator's conv2d fxn

In [5]:
# dry run
np.random.seed(42)
layer1_kernels = np.random.randn(8,1,4,4)*0.01
features = discriminator_conv2d(img_np,layer1_kernels,stride=2,padding=1)
activated_features = leaky_relu(features)
print(f"Input shape: {img_np.shape}")          
print(f"Output features shape: {activated_features.shape}")

Input shape: (1, 64, 64)
Output features shape: (8, 32, 32)


#### loss calculation Binary Cross Entropy(BCE)

In [6]:
def bce_loss(prediction,target):
    epsilon = 1e-12
    prediction = np.clip(prediction,epsilon,1.-epsilon)
    # loss calculation
    loss = -(target*np.log(prediction)+(1-target)*np.log(1-prediction))
    return loss

#### using spectral normalization instead of Batch normalization which is extremely stable than the batch normalization

In [7]:
# Implementation of spectral normalization
# It uses singular value decomposition
def spectral_norm(kernel,u=None,iterations=1):
    out_c = kernel[0]
    W = kernel.reshape(out_c,-1)
    if u is None:
        u = np.random.randn(1,W.shape[0])
        u = u/np.linalg.norm(u)
    for _ in range(iterations):
        # v = u*W
        v = np.dot(u,W) 
        v = v/np.linalg.norm(v)
        # u = v*W.T
        u = np.dot(v,W.T)
        u = u/np.linalg.norm(u)
    sigma = np.dot(np.dot(u,W),v.T)
    normalized_kernel = kernel/sigma
    return normalized_kernel,u


## building discriminator's feet forward according to the spectral normalization

### weights initialization for discriminator feet forward

In [8]:
def discriminator_weights():
    np.random.seed(42)
    weights = {
        'w1' : np.random.randn(8,1,4,4)*0.02,
        'w2' : np.random.randn(16,8,4,4)*0.02,
        'w3' : np.random.randn(32,16,4,4)*0.02,
        'w4' : np.random.randn(64,32,4,4)*0.02,
        'w5' : np.random.randn(1,64,4,4)*0.02
    }
    u_vectors = {
        'u1' : np.random.randn(1,8),
        'u2' : np.random.randn(1,16),
        'u3' : np.random.randn(1,32),
        'u4' : np.random.randn(1,64),
        'u5' : np.random.randn(1,1)
    }
    for i in u_vectors:
        u_vectors[i] = u_vectors[i]/np.linalg.norm(u_vectors[i])
    
    return weights,u_vectors

In [9]:
weights,u_vectors = discriminator_weights()
def discriminator_forward(img_np,weights,u_vectors):
    # layer 1: 64x64 -> 32x32
    sn_w1,u_vectors['u1'] = spectral_norm(weights['w1'],u_vectors['u1'])
    z1 = discriminator_conv2d(img_np,sn_w1,stride=2,padding=1)
    a1 = leaky_relu(z1)
    # layer 2: 32x32 -> 16x16
    sn_w2,u_vectors['u2'] = spectral_norm(weights['w2'],u_vectors['u2'])
    z2 = discriminator_conv2d(a1,sn_w2,stride=2,padding=1)
    a2 = leaky_relu(z2)
    # layer 3: 16x16 -> 8x8
    sn_w3,u_vectors['u3'] = spectral_norm(weights['w3'],u_vectors['u3'])
    z3 = discriminator_conv2d(a2,sn_w3,stride=2,padding=1)
    a3 = leaky_relu(z3)
    # layer 4: 8x8 -> 4x4
    sn_w4,u_vectors['u4'] = spectral_norm(weights['w4'],u_vectors['u4'])
    z4 = discriminator_conv2d(a3,sn_w4,stride = 2,padding=1)
    a4 = leaky_relu(z4)
    # layer 5: 4x4 -> 1x1
    sn_w5,u_vectors['u5'] = spectral_norm(weights['w5'],u_vectors['u5'])
    z5 = discriminator_conv2d(a4,sn_w5,stride=1,padding=0)
    # output layer 0 or 1 (real or fake)
    output = sigmoid(z5)
    return output.flatten()[0],u_vectors


## Generator building

##### here we are using "upsampling+convolution" instead of using "transpose convolution" which reduces the checkerboard artifacts significantly

In [10]:
# upsampling 
def upsample(x):
    return np.repeat(np.repeat(x,2,axis=1),2,axis=2)

In [11]:
# for layer 1 in generator we have to use transpose convolution because a 1x1 cannot be upsampled
def generator_layer1(latent_vector,weights):
    in_c,out_c,kh,kw = weights.shape
    out_h,out_w = 4,4
    output = np.zeros((out_c,out_h,out_w))

    for i in range(in_c):
        for j in range(out_c):
            val = latent_vector[i,0,0]
            output[j] += val*weights[i,j]
    return output

In [12]:
# generator convolution2d function after upsampling
def generator_conv2d(image,kernels,stride=1,padding=1):
        out_c,in_c,kh,kw = kernels.shape
        v_c,v_h,v_w = image.shape
        if padding>0:
            image = np.pad(image,((0,0),(padding,padding),(padding,padding)),mode = 'constant')
        p_h,p_w = image.shape[1],image.shape[2]
        out_h = (p_h-kh)//stride+1
        out_w = (p_w-kw)//stride+1
        output = np.zeros((out_c,out_h,out_w))
        for f in range(out_c):
            for y in range(out_h):
                  for x in range(out_w):
                        y_start = y*stride
                        x_start = x*stride
                        patch = image[:,y_start:y_start+kh,x_start:x_start+kw]
                        output[f,y,x] = np.sum(patch*kernels[f])
        return output

#### weights initialization for generators feed forward

In [13]:
# generator weights initialization
def generator_weights():
    np.random.seed(42)
    weights = {
        'w1' : np.random.randn(100,512,4,4)*0.02,
        'w2' : np.random.randn(256,512,3,3)*0.02,
        'w3' : np.random.randn(128,256,3,3)*0.02,
        'w4' : np.random.randn(64,128,3,3)*0.02,
        'w5' : np.random.randn(1,64,3,3)*0.02
    }
    latent_v = np.random.randn(100,1,1)
    return weights,latent_v


In [ ]:
def generator_forward(latent_v,weights):
    # layer 1: (100,1,1) -> (512,4,4)
    z1 = generator_layer1(latent_v,weights['w1'],stride=1,padding=0)
    a1 = relu(z1)
    # layer 2: (512,8,8) -> (256,8,8)
    a1 = upsample(a1)
    z2 = generator_conv2d(a1,weights['w2'],stride = 1,padding =1)
    a2 = relu(z2)
    # layer 3: (256,16,16) -> (128,16,16)
    a2 = upsample(a2)
    z3 = generator_conv2d(a2,weights['w3'],stride =1, padding =1)
    a3 = relu(z3)
    # layer 4: (128,32,32) -> (64,32,32)
    a3 = upsample(a3)
    z4 = generator_conv2d(a3,weights['w4'],stride =1,padding=1)
    a4 = relu(z4)
    # layer 5: (64,64,64) -> (1,64,64)
    a4 = upsample(a4)
    z5 = generator_conv2d(a4,weights['w5'],stride=1,padding=1)
    y = np.tanh(z5)
    return y
